In [ ]:
import sys, pprint, pandas as pd  
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

import re,pandas as pd
import plotly.io as pio
import plotly.graph_objects as go
import json
from typing import Any, Dict, List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if

from visualization_system.visualization_backend.all_classes import * 



imported


# Initialize 

In [ ]:

# these are just mocks 
def get_config():
    return None 

class DataDrivenStorage:
        
    def __init__( self, config_vars ):
        pass 

    def get_project_dataset(self, project_name=None, filters=None):
        #path =  "../datasets/Demo1/"
        path =  Path("../datasets/IX5I_4P/") 

        inj, prod, locs = self.fetch_data(path) 
        return inj, prod, locs

    def fetch_data(self,path:Path):
        inj  = pd.read_csv(path / "injectors.csv")
        pinj = pd.read_csv(path / "producers.csv")
        locs = pd.read_csv(path / "locations.csv")
        inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
        inj['DAY']   = inj['DATE'].dt.day
        inj['MONTH'] = inj['DATE'].dt.month
        inj['YEAR']  = inj['DATE'].dt.year
        pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
        pinj['DAY']   = pinj['DATE'].dt.day
        pinj['MONTH'] = pinj['DATE'].dt.month
        pinj['YEAR']  = pinj['DATE'].dt.year


        return inj, pinj, locs

def initialize_system( llm ):
   

    #IMPORTS
    from visualization_system.visualization_backend.analyst.semantics.semantic_models import SemanticCatalog, semantic_catalog
    from visualization_system.visualization_backend.analyst.semantics.semantic_models import idioms as all_idiom_rules
    vis_system = AgenticSystem( llm )


    idiom = 'duckdb'
    idiom_rules = all_idiom_rules[idiom]
    semantic_catalog_model = SemanticCatalog.model_validate( semantic_catalog )

    analyst = vis_system.data_analyst_component
    analyst.init_semantic_models( semantic_catalog_model,idiom_rules)
    


    # this mocks data comming from the UI
    # so we just update tge analyst 
    inj,prod,locs = DataDrivenStorage( get_config() ).get_project_dataset(123, {}) 
    analyst.set_data( {'injectors':inj, 'producers':prod, 'locations': locs } )

    return vis_system


llm = azure_llm_if()
vis_system = initialize_system(llm)
vis_system

zero temp, seed 42, top_p = 1


In [ ]:

query = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production since year 2015 for all the wells 
"""

#this is what the presenter consumes 
execution_state = vis_system.run( query )



base_tables=[TableCard(name='injectors', description='Water injection time series. Each row contains a dated observation of water injection for a given Injector well in a given subzone and sector', kind='base', creation_date='2026-06-24 20:48:23.007617', row_count=490, columns=[ColumnCard(name='DATE', data_type='timestamp', description='Injection date.', derived_column=False), ColumnCard(name='NAME', data_type='string', description='Injector well identifier.', derived_column=False), ColumnCard(name='WATER_INJECTION_VOLUME', data_type='float', description='Injected water volume.', derived_column=False), ColumnCard(name='SUBZONE', data_type='string', description='Vertical subzone.', derived_column=False), ColumnCard(name='SECTOR', data_type='integer', description='Geographic sector.', derived_column=False), ColumnCard(name='YEAR', data_type='integer', description='Year from DATE.', derived_column=False), ColumnCard(name='MONTH', data_type='integer', description='Month from DATE.', derive

In [ ]:

presenter = PresenterComponent2( llm )
ui_items = presenter.run( execution_state )
ui_items

processing dataframe result
(4, 2)
instruction Identify the top 5 producer wells based on their cumulative oil production in 2018. Then, calculate the cumulative liquid production (oil + water) for all wells starting from the year 2015 to the most recent data available. Provide the results in a tabular format.


c:\Work\2026\KOC_phase2\AgenticWaterfloodInsights3\notebooks\..\visualization_system\visualization_backend\all_classes.py:1554: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s.dropna(), errors="coerce")


NameError: name 'CHART_AGENT_PROMPT' is not defined

In [12]:
item = ui_items.items[1]
item = item.data['plotly']
pio.show(item)


NameError: name 'ui_items' is not defined